In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
import geopandas as gpd

# =============================================================================
# Kombiniertes Diagramm: NHDA (links) + RA (rechts), gemeinsame Zeilenreihenfolge
#
# Layout (neu):
#   [ NHDA-Balken, gespiegelt ] | [ Label re.-bündig | Regbez | Label li.-bündig ] | [ RA-Balken ]
#
# Datenaufbereitung (Abschnitt 1-3) ist unverändert zur vorherigen Version.
# Geändert wurde nur Abschnitt 4 (Plot).
# =============================================================================

EXPORT_DIR = Path(r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Figures\building_combined")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

NHDA_CSV = Path(r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Figures\building_nhda\landkreis_nhda_buildingtype_stats.csv")
RA_CSV = Path(r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Figures\building_ra\landkreis_ra_buildingtype_stats.csv")

VG250_PATH = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Input\Verwaltungsgebiete\vg250-ew_12-31.utm32s.gpkg.ebenen\vg250-ew_ebenen_1231\DE_VG250.gpkg"

building_types = ['SFH-DB', 'SBD', 'TB', 'MFH-AB', 'unclassified res']
plot_order = ['MFH-AB', 'TB', 'SBD', 'SFH-DB', 'unclassified res']  # MFH-AB liegt an der Mitte an
colors = {
    'SFH-DB': '#82cbec',
    'SBD': '#febd2b',
    'TB': '#9aab4b',
    'MFH-AB': '#d94f21',
    'unclassified res': '#b3b3b3',
}
regbez_map = {
    '091': 'Oberbayern', '092': 'Niederbayern', '093': 'Oberpfalz',
    '094': 'Oberfranken', '095': 'Mittelfranken', '096': 'Unterfranken', '097': 'Schwaben',
}
regbez_order = ['Oberbayern', 'Niederbayern', 'Oberpfalz', 'Oberfranken', 'Mittelfranken', 'Unterfranken', 'Schwaben']


def _clean_name(name):
    n = str(name).strip()
    for p in ['Landkreis ', 'Lkr. ', 'Kreisfreie Stadt ', 'Stadt ', 'Landeshauptstadt ']:
        if n.lower().startswith(p.lower()):
            return n[len(p):].strip()
    return n


def load_raw_shares(csv_path):
    """Lädt eine Stats-CSV und liefert eine ARS-indizierte Tabelle mit
    Rohwerten (nicht normalisiert, nicht gefiltert, nicht sortiert)."""
    base_df = pd.read_csv(csv_path)

    if 'ARS' not in base_df.columns and 'LK_KEY' in base_df.columns:
        base_df = base_df.rename(columns={'LK_KEY': 'ARS'})
    if 'ARS' not in base_df.columns:
        raise ValueError(f'Missing ARS column in {csv_path}')

    if 'county' not in base_df.columns:
        for cand in ['Name', 'lk_name', 'LK_NAME']:
            if cand in base_df.columns:
                base_df = base_df.rename(columns={cand: 'county'})
                break
        else:
            raise ValueError(f'Missing district name column in {csv_path}')

    base_df['ARS'] = base_df['ARS'].astype(str).str.zfill(5)
    base_df['county'] = base_df['county'].astype(str).str.strip()

    if 'lk_type' in base_df.columns:
        district_meta = base_df[['ARS', 'county', 'lk_type']].drop_duplicates(subset=['ARS']).copy()
    else:
        district_meta = base_df[['ARS', 'county']].drop_duplicates(subset=['ARS']).copy()
        district_meta['lk_type'] = pd.NA

    if {'building_type', 'share'}.issubset(base_df.columns):
        long_df = base_df[['ARS', 'county', 'building_type', 'share']].copy()
        long_df['building_type'] = long_df['building_type'].astype(str).str.strip()
        long_df['share'] = pd.to_numeric(long_df['share'], errors='coerce').fillna(0.0)
        long_df = long_df.groupby(['ARS', 'county', 'building_type'], as_index=False)['share'].mean()
        wide = long_df.pivot_table(
            index=['ARS', 'county'], columns='building_type', values='share',
            aggfunc='mean', fill_value=0
        ).reset_index()
    else:
        pct_cols = [f'{bt}_pct' for bt in building_types if f'{bt}_pct' in base_df.columns]
        if not pct_cols:
            raise ValueError(f'No building-type shares found in {csv_path}')
        wide = base_df[['ARS', 'county'] + pct_cols].drop_duplicates(subset=['ARS']).copy()
        wide = wide.rename(columns={f'{bt}_pct': bt for bt in building_types if f'{bt}_pct' in wide.columns})

    for bt in building_types:
        if bt not in wide.columns:
            wide[bt] = 0.0
        wide[bt] = pd.to_numeric(wide[bt], errors='coerce').fillna(0.0)

    return wide.set_index('ARS'), district_meta


# --- 1) NHDA: unveränderte Master-Logik ----------------------------------------
wide_nhda_raw, district_meta_nhda = load_raw_shares(NHDA_CSV)
wide_nhda_raw = wide_nhda_raw.reset_index()

vg250 = gpd.read_file(VG250_PATH, layer="v_vg250_krs")
vg250.columns = vg250.columns.str.strip()
ars_col = next((c for c in ["ARS", "Regionalschlüssel_ARS", "RegioschlüsselAufgefüllt", "RS"] if c in vg250.columns), None)
if ars_col is None:
    raise ValueError(f"Keine ARS-Spalte gefunden. Spalten: {vg250.columns.tolist()}")
vg250 = vg250.rename(columns={ars_col: "ARS"})
vg250_meta = vg250[["ARS", "Bezeichnung"]].copy()
vg250_meta["ARS"] = vg250_meta["ARS"].astype(str).str.zfill(5)

district_meta_nhda["ARS"] = district_meta_nhda["ARS"].astype(str).str.zfill(5)
district_meta_nhda = district_meta_nhda.merge(vg250_meta.drop_duplicates(subset=["ARS"]), on="ARS", how="left")

wide_nhda = wide_nhda_raw.copy()
wide_nhda['sum_share'] = wide_nhda[building_types].sum(axis=1)
has_nhda = wide_nhda['sum_share'] > 0
removed_no_nhda = int((~has_nhda).sum())
wide_nhda = wide_nhda.loc[has_nhda].copy()
if wide_nhda.empty:
    raise ValueError('After filtering, no districts with NHDA remain to plot.')

wide_nhda[building_types] = wide_nhda[building_types].div(wide_nhda['sum_share'], axis=0) * 100
wide_nhda['sum_final'] = wide_nhda[building_types].sum(axis=1)
wide_nhda['SFH-DB'] = wide_nhda['SFH-DB'] + (100.0 - wide_nhda['sum_final'])

print(f'Excluded districts without NHDA: {removed_no_nhda}')

wide_nhda['regbez_code'] = wide_nhda['ARS'].str[:3]
wide_nhda['regbez'] = wide_nhda['regbez_code'].map(regbez_map)
wide_nhda = wide_nhda.merge(district_meta_nhda[['ARS', 'Bezeichnung']].drop_duplicates(subset=['ARS']), on='ARS', how='left')
wide_nhda = wide_nhda.merge(district_meta_nhda[['ARS', 'lk_type']].drop_duplicates(subset=['ARS']), on='ARS', how='left')

wide_nhda['base_name'] = wide_nhda['county'].apply(_clean_name)
ambiguous_names = set(wide_nhda['base_name'].value_counts()[lambda s: s > 1].index)


def _label(name, bezeichnung, name_is_ambiguous):
    n = _clean_name(name)
    b = '' if pd.isna(bezeichnung) else str(bezeichnung).strip().lower()
    if name_is_ambiguous and b == 'landkreis':
        return f'{n} (LK)'
    return n


wide_nhda['district_label'] = wide_nhda.apply(
    lambda r: _label(r['county'], r['Bezeichnung'], r['base_name'] in ambiguous_names), axis=1
)

wide_nhda['regbez'] = pd.Categorical(wide_nhda['regbez'], categories=regbez_order, ordered=True)
wide_nhda = wide_nhda.sort_values(['regbez', 'district_label']).reset_index(drop=True)

# Das ist jetzt die EINZIGE gültige Zeilen-Reihenfolge für die gesamte Abbildung.
master_order = wide_nhda[['ARS', 'district_label', 'regbez']].copy()

# --- 2) RA: an die Master-Reihenfolge andocken (keine eigene Filterung/Sortierung!) --
wide_ra_raw, _ = load_raw_shares(RA_CSV)
wide_ra_raw = wide_ra_raw.reset_index()
wide_ra_raw['sum_share'] = wide_ra_raw[building_types].sum(axis=1)

# Normalisieren, aber NICHT filtern/sortieren -- Reihenfolge kommt von master_order
mask = wide_ra_raw['sum_share'] > 0
wide_ra_raw.loc[mask, building_types] = wide_ra_raw.loc[mask, building_types].div(
    wide_ra_raw.loc[mask, 'sum_share'], axis=0
) * 100
wide_ra_raw.loc[mask, 'SFH-DB'] = wide_ra_raw.loc[mask, 'SFH-DB'] + (100.0 - wide_ra_raw.loc[mask, building_types].sum(axis=1))

wide_ra = master_order.merge(
    wide_ra_raw[['ARS'] + building_types], on='ARS', how='left'
)
missing_ra = wide_ra[building_types].isna().all(axis=1)
n_missing_ra = int(missing_ra.sum())
if n_missing_ra:
    print(f'Warnung: {n_missing_ra} Landkreis(e) aus der NHDA-Reihenfolge haben keine RA-Daten -> leerer Balken.')
wide_ra[building_types] = wide_ra[building_types].fillna(0.0)

# Reihenfolge exakt wie master_order (per Definition schon so, da merge left-join auf master_order)
assert list(wide_ra['ARS']) == list(wide_nhda['ARS']), 'Reihenfolge zwischen RA und NHDA weicht ab!'

# --- 3) Median-Tabellen (getrennt, wie in den Originalskripten) ----------------
median_nhda = wide_nhda.groupby('regbez', observed=False)[building_types].median().reindex(regbez_order).dropna(how='all').round(2)
median_ra = wide_ra.groupby('regbez', observed=False)[building_types].median().reindex(regbez_order).dropna(how='all').round(2)

print('\nMedian NHDA (%) je Regierungsbezirk:')
display(median_nhda)
print('\nMedian RA (%) je Regierungsbezirk:')
display(median_ra)

median_nhda.to_csv(EXPORT_DIR / 'median_building_type_ratios_nhda.csv', encoding='utf-8-sig')
median_ra.to_csv(EXPORT_DIR / 'median_building_type_ratios_ra.csv', encoding='utf-8-sig')

# --- 4) Kombiniertes Diagramm: NHDA links | Regbez-Mitte | RA rechts -----------
line_spacing = 1.25
n = len(wide_nhda)
y = [i * line_spacing for i in range(n)]

fig, (ax_nhda, ax_mid, ax_ra) = plt.subplots(
    1, 3, figsize=(12.5, 11.69 * line_spacing), sharey=True,
    gridspec_kw={'width_ratios': [1.0, 0.16, 1.0], 'wspace': 0.0}
)

present_types = [t for t in plot_order if (wide_nhda[t].sum() > 0) or (wide_ra[t].sum() > 0)]

# --- linke Seite: NHDA, gespiegelt (negative Werte, wächst nach links) ---
left = pd.Series(0.0, index=wide_nhda.index)
for t in present_types:
    vals = wide_nhda[t]
    ax_nhda.barh(y, -vals, left=-left, height=0.82, color=colors.get(t, '#7f7f7f'),
                 edgecolor='white', linewidth=0.4, label=t)
    left = left + vals

# --- rechte Seite: RA, normal (wächst nach rechts) ---
left = pd.Series(0.0, index=wide_ra.index)
for t in present_types:
    vals = wide_ra[t]
    ax_ra.barh(y, vals, left=left, height=0.82, color=colors.get(t, '#7f7f7f'),
               edgecolor='white', linewidth=0.4)
    left = left + vals

# Achsen NHDA (links, gespiegelt)
ax_nhda.set_xlim(-112, 0)
ax_nhda.set_xticks(range(-100, 1, 20))
ax_nhda.set_xticklabels([str(abs(v)) for v in range(-100, 1, 20)])
ax_nhda.set_xlabel('NHDA – Anteil (%)', fontsize=8)
ax_nhda.tick_params(axis='x', labelsize=8.5)
ax_nhda.spines['top'].set_visible(False)
ax_nhda.spines['left'].set_visible(False)
ax_nhda.spines['right'].set_visible(False)
ax_nhda.grid(axis='x', linestyle='--', alpha=0.35, linewidth=0.6)
ax_nhda.set_axisbelow(True)
ax_nhda.set_yticks(y)
ax_nhda.set_yticklabels(wide_nhda['district_label'], fontsize=7.5)
ax_nhda.tick_params(axis='y', length=0, labelleft=True, labelright=False, pad=4)

# Achsen RA (rechts, normal)
ax_ra.set_xlim(0, 112)
ax_ra.set_xticks(range(0, 101, 20))
ax_ra.set_xlabel('RA – Anteil (%)', fontsize=8)
ax_ra.tick_params(axis='x', labelsize=8.5)
ax_ra.spines['top'].set_visible(False)
ax_ra.spines['left'].set_visible(False)
ax_ra.spines['right'].set_visible(False)
ax_ra.grid(axis='x', linestyle='--', alpha=0.35, linewidth=0.6)
ax_ra.set_axisbelow(True)
ax_ra.yaxis.tick_right()
ax_ra.set_yticks(y)
ax_ra.set_yticklabels(wide_nhda['district_label'], fontsize=7.5)
ax_ra.tick_params(axis='y', length=0, labelright=True, labelleft=False, pad=4)

# --- Mittelspalte: Landkreis-Label (doppelt) + Regierungsbezirk --------------
ax_mid.set_xlim(-1, 1)
ax_mid.set_xticks([])
ax_mid.tick_params(axis='y', left=False, right=False, labelleft=False, labelright=False)
for spine in ax_mid.spines.values():
    spine.set_visible(False)

label_inset = 0.06  # nur noch relevant falls später wieder Text in der Mittelspalte gebraucht wird

for ax in (ax_nhda, ax_mid, ax_ra):
    ax.invert_yaxis()
    ax.margins(y=0)
    ax.set_ylim((n - 0.5) * line_spacing, -0.5 * line_spacing)

# Regierungsbezirk-Trennlinien (durchgängig über alle drei Panels) + Name mittig
group_sizes = wide_nhda['regbez'].value_counts(sort=False).reindex(regbez_order).fillna(0).astype(int)
start = 0
for rb, size in group_sizes.items():
    if size == 0:
        continue
    end = start + size - 1
    center = (y[start] + y[end]) / 2
    ax_mid.text(0, center, rb, va='center', ha='center', rotation=90,
                fontsize=7.5, fontweight='bold')
    if end < n - 1:
        line_y = (y[end] + y[end + 1]) / 2
        for ax in (ax_nhda, ax_mid, ax_ra):
            ax.axhline(line_y, color='#444444', linewidth=1.6, alpha=0.9)
    start = end + 1

ax_nhda.set_title('NHDA', fontsize=11, fontweight='bold', loc='center')
ax_ra.set_title('RA', fontsize=11, fontweight='bold', loc='center')

ax_ra.legend(
    title='Building Type', ncol=1, loc='upper left', bbox_to_anchor=(1.02, 1.0),
    frameon=False, fontsize=8.5, title_fontsize=7, borderaxespad=0.0,
    handlelength=1.2, labelspacing=0.25, borderpad=0.0, columnspacing=0.6,
    handletextpad=0.4, markerscale=0.8, prop={'size': 8.5}
)

fig.subplots_adjust(left=0.14, right=0.80, top=0.96, bottom=0.045, wspace=0.0)

out_file = EXPORT_DIR / 'nhda_ra_res_ratio_combined_horizontal.jpg'
plt.savefig(out_file, dpi=300, bbox_inches='tight', facecolor='white')
print(f'Saved: {out_file}')

plt.show()